# 30b — Light-artifact diagnostics (uniform)

`30` shows WT (light-only control) shifting PSE under opto, larger than HET. Before any curve
parameter from `30`/`31` is reportable, this asks whether that WT effect is a
**stimulus-independent side bias** (light-delivery artifact → discard PSE/slope) or a
stimulus-dependent decision change.

Model-free: empirical P(choose B) by stimulus, opto vs the interleaved `opto_off`, per animal.

- **Primary (§3–§4).** ΔP(B) vs stimulus. Flat & non-zero *including the saturated tails* ⇒
  additive offset = artifact. Peaked near the boundary, ~0 in the tails ⇒ criterion shift = real.
- **Secondary (§5).** SDT criterion `c` / `d′` — conventional language only; `c` carries the
  same horizontal-shift ambiguity as the psychometric μ, so it is **not** the discriminator.
- **Control (§6).** Is recency (the history channel) spared? If PSE shifts but recency doesn't,
  the history-stat results in `30`/`31` stand and only PSE/slope are set aside.

Uniform only: the tail and SDT-rate tests need both stimulus extremes sampled, which the Hard
distributions under-sample. n=4 WT floors the genotype contrast at ≈0.057 — the *shape* and
sign-consistency are the evidence, not a p-value.

In [ ]:
%matplotlib inline
import numpy as np, pandas as pd
from shared_setup import *

from analysis.opto import (
    compute_choice_by_stimulus, compute_side_bias, compute_sdt, extract_opto_estimates,
)
from behav_utils.analysis import paired_diff, rank_test, min_achievable_p
from plotting.opto import plot_choice_by_stimulus, plot_delta_by_stimulus, plot_delta_swarm

PHASE = 'uniform'
experiment, info = load_data()
geno, groups = gather_genotypes(experiment)
het, wt = groups.get('het', []), groups.get('wt', [])
opto_ids = het + wt
if not het or not wt:
    raise RuntimeError("No HET/WT genotypes on loaded animals (gather_genotypes reads .genotype).")
print(f"loaded ({info['mode']}): het={het} wt={wt}")

In [ ]:
# One model-free pass: empirical P(B) by stimulus + hit/FA, opto vs opto_off, per animal.
cbs = compute_choice_by_stimulus(experiment, phase=PHASE, animals=opto_ids)
present = sorted(cbs['binned']['animal'].unique())
print("conditions:", cbs['trial_types'], "| animals with data:", present)

## §2 · Empirical psychometric, opto vs opto_off — per WT animal
Raw binned P(choose B), no fitted curve. Watch the **tails**: an additive offset lifts them, a
criterion shift leaves them pinned at floor/ceiling.

In [ ]:
wt_present = [a for a in wt if a in set(cbs['binned']['animal'])]
ncol = min(4, len(wt_present)) or 1
fig, axes = plt.subplots(1, ncol, figsize=(4 * ncol, 3.6), squeeze=False)
for ax, aid in zip(axes[0], wt_present):
    plot_choice_by_stimulus(cbs, aid, ax=ax)
fig.tight_layout()

## §3 · ΔP(B) vs stimulus — the discriminator
opto − opto_off at each stimulus level, one thin line per animal, mean in red.
**Flat & non-zero (incl. tails) ⇒ additive offset (light artifact). Peaked near 0, ~0 in the
tails ⇒ criterion/boundary shift (real).**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
plot_delta_by_stimulus(cbs, ax=axes[0], genotype='wt')
plot_delta_by_stimulus(cbs, ax=axes[1], genotype='het')
fig.tight_layout()

## §4 · Side-bias scalars (primary)
`net_bias` = overall ΔP(B); `tail_delta` = mean ΔP(B) in the saturated bins; `boundary_delta` =
near-threshold bins. **Artifact ⇒ tail_delta and boundary_delta both non-zero (flat).
Criterion shift ⇒ boundary_delta only, tail_delta ≈ 0.** `net_bias` het vs wt tests a lateralised
bias, reported with its achievable-p floor (n=4 WT ⇒ floor ≈ 0.057).

In [ ]:
sb = compute_side_bias(cbs)
display(sb.round(3))

h = sb[sb.genotype == 'het']['net_bias'].dropna().to_numpy()
w = sb[sb.genotype == 'wt']['net_bias'].dropna().to_numpy()
res = rank_test(h, w, paired=False)
print(f"net_bias het vs wt: U={res['statistic']:.1f}  p={res['p']:.3g}  "
      f"(min achievable p={min_achievable_p('rank_sum', n1=len(h), n2=len(w)):.3g}, "
      f"n_het={len(h)} n_wt={len(w)})")

long = sb.melt(id_vars=['animal', 'genotype'],
               value_vars=['net_bias', 'tail_delta', 'boundary_delta'],
               var_name='stat', value_name='delta')
fig, axes = plt.subplots(1, 3, figsize=(11, 3.6))
for ax, st in zip(axes, ['net_bias', 'tail_delta', 'boundary_delta']):
    plot_delta_swarm(long, st, ax=ax, p_value=res['p'] if st == 'net_bias' else None)
fig.tight_layout()

## §5 · Signal-detection (secondary, descriptive)
Criterion `c` and sensitivity `d′` per condition (empirical hit/FA, log-linear correction).
**Not the discriminator** — `c` absorbs both a motor offset and a boundary move, the same
ambiguity as PSE. Reported as conventional language beside §3–§4. `saturated` flags animals whose
raw hit/fa hit 0/1 (`d′` imprecise there). Equal-variance Gaussian assumed.

In [ ]:
sdt = compute_sdt(cbs)
display(sdt.round(3))
if sdt['saturated'].any():
    print("saturated (d′ imprecise):", sorted(sdt[sdt.saturated]['animal']))

long_sdt = sdt.dropna(subset=['d_c']).melt(
    id_vars=['animal', 'genotype'], value_vars=['d_c', 'd_dprime'],
    var_name='stat', value_name='delta')
fig, axes = plt.subplots(1, 2, figsize=(7.5, 3.6))
for ax, st in zip(axes, ['d_c', 'd_dprime']):
    plot_delta_swarm(long_sdt, st, ax=ax)
fig.tight_layout()

## §6 · Is recency spared? (history-channel control)
The payoff. A pure offset artifact should not change recency (trial-to-trial updating). If PSE
shifts but recency doesn't, the history stats in `30`/`31` stand and only PSE/slope are binned.

In [ ]:
rec = extract_opto_estimates(experiment, phases=PHASE, stats=['recency'],
                             trial_types=('opto', 'opto_off'), animals=opto_ids)
drec = paired_diff(rec, by='trial_type', a='opto', b='opto_off')   # opto − opto_off per animal

hr = drec[drec.genotype == 'het']['delta'].dropna().to_numpy()
wr = drec[drec.genotype == 'wt']['delta'].dropna().to_numpy()
rr = rank_test(hr, wr, paired=False)
print(f"recency Δ het vs wt: U={rr['statistic']:.1f}  p={rr['p']:.3g}  "
      f"(min achievable p={min_achievable_p('rank_sum', n1=len(hr), n2=len(wr)):.3g})")
fig, ax = plt.subplots(figsize=(3.4, 3.6))
plot_delta_swarm(drec, 'recency', ax=ax, p_value=rr['p'])
fig.tight_layout()

**Reading it.** Discriminator is §3–§4: if ΔP(B) is flat and non-zero into the tails and
`tail_delta`≈`boundary_delta`≈`net_bias` with a consistent sign across WT, the WT effect is a
light-delivery side bias → **bin PSE/slope from `30`/`31`**. If instead ΔP(B) peaks at the
boundary with `tail_delta`≈0, it is a criterion shift and the interpretation is different. §5 is
description only. §6 decides the history stats: recency Δ ≈ 0 (both genotypes) ⇒ the history
channel is clean and the `30`/`31` history-stat conclusions hold. All descriptive at this n.